# Contextual Recommendation System for News Articles (Word2Vec)

**Objective**: Build a recommendation system that suggests relevant news articles based on contextual similarity of content using Word2Vec embeddings.

This notebook follows the exact steps in your project specification:
1. **Preprocess and Prepare Text Data** (tokenize, clean, remove stopwords & punctuation)
2. **Train / Use a Word2Vec Model** and build an **article embedding** by averaging word vectors
3. **Compute Cosine Similarity** between article embeddings and **recommend top-N** similar articles

---

- Uses a **standard dataset**: 20 Newsgroups (via scikit-learn). If download is not available, it **falls back** to a tiny in-notebook sample so students can still run the pipeline end-to-end.
- Clear, modular code with **functions** you can reuse.
- Includes a **t-SNE visualization** of article embeddings.
- Keeps the Word2Vec model small (trained on the chosen corpus) so it runs on laptops.

⚠️ *Note*: The pre-trained Google News Word2Vec model (~1.5GB) is intentionally **not** used to keep things light. You can plug it in later if desired.


## 0) Setup
Install and import libraries. If you're in an offline environment, skip the pip cell if already installed.

In [ ]:
# If needed, uncomment to install dependencies
# !pip install gensim nltk scikit-learn matplotlib

In [ ]:
import re
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_20newsgroups
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.manifold import TSNE
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from gensim.models import Word2Vec
import nltk

try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')


## 1) Load a Standard Dataset (20 Newsgroups) with Fallback
We try to load the **20 Newsgroups** training subset. If downloading fails, we fall back to a **tiny sample corpus** embedded in this notebook so that the rest of the pipeline still works  

In [ ]:
def load_dataset():
    try:
        data = fetch_20newsgroups(subset='train', remove=('headers','footers','quotes'))
        texts = data.data
        targets = data.target
        target_names = data.target_names
        print(f"Loaded 20 Newsgroups: {len(texts)} documents across {len(target_names)} categories.")
        return texts, targets, target_names
    except Exception as e:
        print("Could not load 20 Newsgroups (offline?). Using a small fallback corpus.")
        texts = [
            "NASA announces new mission to study Mars atmosphere and search for water clues.",
            "Local team wins the regional basketball championship after a thrilling final.",
            "Tech giants unveil AI tools for developers focusing on code completion and debugging.",
            "Economists warn of inflation risks as central bank considers interest rate changes.",
            "Astronomers discover exoplanet in habitable zone with potential signs of water.",
            "Basketball league introduces new rules to improve game pace and reduce fouls.",
            "New research shows deep learning models outperform traditional methods in image tasks.",
            "Central bank raises interest rates to curb inflation amid strong job market.",
        ]
        targets = np.array([0,1,2,3,0,1,2,3])
        target_names = ["space","sports","technology","economy"]
        return texts, targets, target_names

texts, targets, target_names = load_dataset()

## 2) Text Preprocessing
Steps:
- Lowercase
- Remove punctuation/numbers
- Tokenize
- Remove stopwords

We use scikit-learn's **ENGLISH_STOP_WORDS** (no external downloads) and a simple regex tokenizer. You can switch to NLTK stopwords if desired.

In [ ]:
STOPWORDS = set(ENGLISH_STOP_WORDS)
TOKEN_RE = re.compile(r"[A-Za-z]+")

def preprocess(text):
    tokens = TOKEN_RE.findall(text.lower())
    tokens = [t for t in tokens if t not in STOPWORDS and len(t) > 2]
    return tokens

tokenized_corpus = [preprocess(t) for t in texts]
print(tokenized_corpus[0][:30])

## 3) Train a Word2Vec Model
We train a small **Word2Vec** model using **Gensim** on our cleaned corpus. For teaching, small dimensions are enough and train quickly on CPU.

In [ ]:
w2v_dim = 100
w2v = Word2Vec(
    sentences=tokenized_corpus,
    vector_size=w2v_dim,
    window=5,
    min_count=2,
    workers=2,
    sg=1,  # skip-gram works well for semantics
    epochs=10
)
print(f"Vocab size: {len(w2v.wv)}")

## 4) Build Article Embeddings (Average of Word Embeddings)
For each article, we average the Word2Vec vectors of the words present in the model's vocabulary.

In [ ]:
def article_vector(tokens, model, dim):
    vecs = [model.wv[w] for w in tokens if w in model.wv]
    if not vecs:
        return np.zeros(dim)
    return np.mean(vecs, axis=0)

article_embeddings = np.vstack([article_vector(toks, w2v, w2v_dim) for toks in tokenized_corpus])
article_embeddings.shape

## 5) Compute Similarities & Build a Recommender
We compute a cosine similarity matrix and create a function to fetch top-N similar articles for a given article index.

In [ ]:
sim_matrix = cosine_similarity(article_embeddings)

def title_from_text(text, max_words=12):
    words = text.strip().split()
    return " ".join(words[:max_words]) + ("..." if len(words) > max_words else "")

def recommend(article_id, top_n=5):
    sims = sim_matrix[article_id]
    # Exclude self by setting its similarity to -inf
    sims = sims.copy()
    sims[article_id] = -np.inf
    idx = np.argsort(-sims)[:top_n]
    return idx, sims[idx]

# Demo: pick a random article and recommend
np.random.seed(0)
example_id = np.random.randint(0, len(texts))
idxs, scores = recommend(example_id, top_n=5)
print("Query Article:")
print(title_from_text(texts[example_id]))
print("\nRecommendations:")
for i, (j, s) in enumerate(zip(idxs, scores), start=1):
    print(f"{i}. (score={s:.3f}) {title_from_text(texts[j])}")

## 6) Visualize Article Embeddings with t-SNE
We project the article embeddings to 2D using t-SNE and scatter-plot them. If the dataset is large, you can **sample** for speed.

In [ ]:
max_points = 600  # for speed on large datasets
indices = np.arange(len(article_embeddings))
if len(indices) > max_points:
    np.random.shuffle(indices)
    indices = indices[:max_points]

emb_sample = article_embeddings[indices]
y_sample = np.array(targets)[indices]

tsne = TSNE(n_components=2, init='random', perplexity=30, learning_rate='auto', n_iter=1000, verbose=1)
emb_2d = tsne.fit_transform(emb_sample)

plt.figure(figsize=(7,5))
plt.scatter(emb_2d[:,0], emb_2d[:,1], s=10)
plt.title("t-SNE of Article Embeddings (sample)")
plt.xlabel("Dim 1")
plt.ylabel("Dim 2")
plt.show()

## 7) Wrap the Recommender Demos
Helper to print recommendations given any article index. You can loop over a few examples in class.

In [ ]:
def show_recommendations(article_id, k=5):
    print("\n=== QUERY ===")
    print(title_from_text(texts[article_id], 30))
    idxs, scores = recommend(article_id, top_n=k)
    print("\n=== RECOMMENDATIONS ===")
    for rank, (j, s) in enumerate(zip(idxs, scores), start=1):
        cat_q = target_names[targets[article_id]] if 0 <= targets[article_id] < len(target_names) else "n/a"
        cat_j = target_names[targets[j]] if 0 <= targets[j] < len(target_names) else "n/a"
        print(f"{rank}. [score={s:.3f}] ({cat_j}) {title_from_text(texts[j], 30)}")

# Try it
show_recommendations(example_id, k=5)

## 8) (Optional) Swap in a Pre-trained Word2Vec
If you want stronger semantics and have the resources, you can load a pre-trained model such as **Google News Word2Vec** and skip training. Example code (commented to keep this notebook light):

In [ ]:
# from gensim.models import KeyedVectors
# # Download binary from https://code.google.com/archive/p/word2vec/ (about 1.5GB)
# google_news_path = "/path/to/GoogleNews-vectors-negative300.bin"
# kv = KeyedVectors.load_word2vec_format(google_news_path, binary=True)
# w2v_dim = kv.vector_size
# def article_vector_pretrained(tokens, kv, dim):
#     vecs = [kv[w] for w in tokens if w in kv]
#     return np.mean(vecs, axis=0) if vecs else np.zeros(dim)
# article_embeddings = np.vstack([article_vector_pretrained(t, kv, w2v_dim) for t in tokenized_corpus])
# sim_matrix = cosine_similarity(article_embeddings)